<a href="https://colab.research.google.com/github/chiemahp/Flyrank-internship-ml/blob/main/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chiemahp/Flyrank-internship-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, f1_score

repo_root = Path.cwd()
raw_path = repo_root / "data" / "raw" / "content_refresh_anonymized.csv"
for parent in [repo_root, *repo_root.parents]:
    candidate = parent / "data" / "raw" / "content_refresh_anonymized.csv"
    if candidate.exists():
        raw_path = candidate
        repo_root = parent
        break

df = pd.read_csv(raw_path)

for col in [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d",
    "users_90d", "engaged_sessions_90d", "ai_sessions_90d",
    "scroll_events_90d", "days_with_impressions", "days_with_sessions",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "content_age_days", "age_tier_order", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate",
    "ai_traffic_pct", "trend_pct",
]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

for col in [
    "competition_level", "content_type", "main_intent", "provider_used",
    "model_used", "age_tier", "freshness_tier", "word_count_tier",
    "char_count_tier", "impression_tier", "position_tier", "trend_direction",
]:
    df[col] = df[col].fillna("unknown").astype(str).replace({"": "unknown", "nan": "unknown"})

df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])

df["has_clicks"] = (df["clicks_90d"] > 0).astype(int)
df["has_ai_sessions"] = (df["ai_sessions_90d"] > 0).astype(int)
df["measurable_opportunity"] = ((df["impressions_90d"] >= 100) & (df["sessions_90d"] > 0)).astype(int)

numeric_features = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d",
    "log_ai_sessions_90d", "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update", "ctr", "avg_position",
    "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
categorical_features = [
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
]

X_num = df[numeric_features].apply(pd.to_numeric, errors="coerce").fillna(0)
X_cat = df[categorical_features].fillna("unknown").astype(str)
X = pd.concat([X_num, pd.get_dummies(X_cat, prefix=categorical_features, dtype=float)], axis=1)
y = df["is_declining_label"].astype(int)

print(f"Prepared rows: {len(df):,}")
print(f"Target positive rate: {y.mean():.3f}")
X.head()

Prepared rows: 30,000
Target positive rate: 0.542


,search_volume,competition,cpc,word_count,char_count,log_impressions_90d,log_clicks_90d,log_sessions_90d,log_ai_sessions_90d,days_with_impressions,...,word_count_tier_unknown,impression_tier_excellent,impression_tier_good,impression_tier_low,impression_tier_moderate,position_tier_deep,position_tier_page_1,position_tier_page_3_5,position_tier_striking,position_tier_top_3
0,10.0,0.67,2.05,3221.0,20457.0,8.243808,3.401197,2.890372,0.0,88,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,90.0,0.01,0.05,2481.0,15562.0,9.636980,2.079442,2.302585,0.0,88,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,0.0,0.00,0.00,3515.0,23643.0,9.440023,2.484907,2.484907,0.0,88,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,10.0,0.00,0.00,0.0,0.0,9.371779,4.077537,4.369448,0.0,88,...,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,0.0,0.00,0.00,2803.0,17469.0,9.859588,3.218876,4.983607,0.0,88,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [ ]:
# Use a client-aware holdout to mirror the repository's honest split strategy.
client_series = df["client_id"].fillna("unknown").astype(str)
unique_clients = client_series.drop_duplicates().to_numpy()
random_generator = np.random.default_rng(42)
shuffled_clients = random_generator.permutation(unique_clients)
test_client_count = max(1, int(round(len(shuffled_clients) * 0.2)))
test_clients = set(shuffled_clients[:test_client_count])

test_mask = client_series.isin(test_clients).to_numpy()
train_mask = ~test_mask

train_index = df.index[train_mask]
test_index = df.index[test_mask]

print(f"Train rows: {len(train_index):,}")
print(f"Test rows: {len(test_index):,}")
print(f"Split strategy: client_holdout")

Train rows: 27,675
Test rows: 2,325
Split strategy: client_holdout


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Baseline score: use the same baseline_refresh_score from the prior notebook as a simple ranked score.
# We compare a learned logistic model against that baseline at the same test split.
# The baseline is represented by the raw ranking score on the test rows.
baseline_score = df.loc[test_index, "impressions_90d"].rank(pct=True).to_numpy()

model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
])

model.fit(X.iloc[train_index], y.iloc[train_index])
model_prob = model.predict_proba(X.iloc[test_index])[:, 1]

# Turn probabilities into binary predictions at 0.5.
model_pred = (model_prob >= 0.5).astype(int)
baseline_pred = (baseline_score >= 0.5).astype(int)

results = pd.DataFrame([
    {
        "model": "baseline",
        "roc_auc": roc_auc_score(y.iloc[test_index], baseline_score),
        "average_precision": average_precision_score(y.iloc[test_index], baseline_score),
        "precision": precision_score(y.iloc[test_index], baseline_pred, zero_division=0),
        "recall": recall_score(y.iloc[test_index], baseline_pred, zero_division=0),
        "f1": f1_score(y.iloc[test_index], baseline_pred, zero_division=0),
    },
    {
        "model": "logistic_regression",
        "roc_auc": roc_auc_score(y.iloc[test_index], model_prob),
        "average_precision": average_precision_score(y.iloc[test_index], model_prob),
        "precision": precision_score(y.iloc[test_index], model_pred, zero_division=0),
        "recall": recall_score(y.iloc[test_index], model_pred, zero_division=0),
        "f1": f1_score(y.iloc[test_index], model_pred, zero_division=0),
    },
])
results

,model,roc_auc,average_precision,precision,recall,f1
0,baseline,0.697239,0.498826,0.545611,0.697470,0.612265
1,logistic_regression,0.700291,0.521542,0.565934,0.566557,0.566245


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# Inspect the top features from the trained model.
coef_frame = pd.DataFrame({
    "feature": X.columns,
    "coefficient": model.named_steps["model"].coef_[0],
})
coef_frame = coef_frame.sort_values("coefficient", key=lambda s: s.abs(), ascending=False).head(15)
coef_frame

,feature,coefficient
5,log_impressions_90d,1.658755
3,word_count,1.607656
4,char_count,-1.351979
6,log_clicks_90d,-0.656016
14,avg_position,-0.404586
9,days_with_impressions,-0.251329
11,content_age_days,-0.248888
45,impression_tier_low,0.245641
7,log_sessions_90d,-0.243358
51,position_tier_top_3,-0.190942


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.